# 03 Starship and Orbital Compute Options

Starship and orbital compute are valued as probability-weighted options,
not base-case cash flows. The base case should only include option value
supported by milestones or clear internal economics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/spacex_ipo/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/spacex_ipo/data").exists()
)
USD_BN = 1_000_000_000
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
options

In [ ]:
option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
starship_grid = build_sensitivity_grid(
    row_values=[0.10, 0.25, 0.40, 0.55, 0.70],
    column_values=[25, 50, 100, 150, 200],
    formula=lambda probability, cadence: probability * cadence * 2.5,
    row_name="Starship success probability",
    column_name="Reliable launches per year",
)
starship_grid

In [ ]:
cost_payload_grid = build_sensitivity_grid(
    row_values=[100, 250, 500, 1_000],
    column_values=[50, 100, 200, 400],
    formula=lambda cost_per_kg, demand_kt: demand_kt * 1_000_000 / cost_per_kg / 1_000,
    row_name="Cost per kg ($)",
    column_name="Payload demand (kt)",
)
cost_payload_grid